# Tool Nodes in LangGraph Agents

A hand-written `while` loop can implement tool calling — but as agents grow, you need **checkpointing**, **conditional routing**, **streaming**, and **visible control flow**. **LangGraph** encodes the ReAct pattern as a **state graph**:

```
START → model → tools_condition ──► tools → model → … → END
                      │
                      └── END (no tool_calls on last message)
```

Two prebuilt pieces do most of the work:

- **`ToolNode`** — executes all `tool_calls` on the last `AIMessage` and appends `ToolMessage` results.
- **`tools_condition`** — routes to `"tools"` if the last message has `tool_calls`, otherwise to `END`.

This notebook builds that graph end to end using an **Order Support API** (order lookup, policy search, product catalog). You will run **`ToolNode` in isolation**, compile a full ReAct graph, compare it to a manual loop, add **step limits** and **audit logging**, and test without a live LLM.

Everything is explained here — no prior notebooks required. An optional cell runs against OpenAI when you set `USE_LIVE_LLM = True`.

In [ ]:
"""
Order Support API + LangGraph tool-node setup.
"""

import json
import os
import re
import uuid
from typing import Annotated, Any, Literal

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


# --- In-memory backend ---
PRODUCT_CATALOG = [
    {"sku": "LAP-001", "name": "Pro Laptop 14", "price": 1299.00, "stock": 12},
    {"sku": "MON-002", "name": "4K Monitor 27", "price": 449.00, "stock": 34},
    {"sku": "KEY-003", "name": "Mechanical Keyboard", "price": 129.00, "stock": 0},
]

ORDERS = {
    "ORD-1001": {"status": "shipped", "tracking": "1Z999AA10123456784", "customer": "alice@shop.com"},
}

POLICY_DOCS = [
    {"id": "ship-01", "text": "Standard shipping 5-7 business days. Express 2 days for $15."},
    {"id": "ret-02", "text": "Returns within 30 days if unused. Refunds in 5 business days."},
]

print("LangGraph tool-node environment ready.")

---

## 1. Define Tools with `@tool`

LangGraph's `ToolNode` accepts a list of LangChain **`@tool`** functions. Each tool gets a name, description, and JSON Schema for arguments — the same schemas the LLM sees when you call `bind_tools`.

In [ ]:
@tool
def lookup_order(order_id: str) -> str:
    """Look up order status and tracking by order ID (e.g. ORD-1001). Use when customer provides an order ID."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return json.dumps({"found": False, "error": f"Order {order_id} not found"})
    return json.dumps({"found": True, "order_id": order_id.upper(), **order})


@tool
def search_policy(query: str) -> str:
    """Search shipping and returns policy docs. Use for refund/shipping questions without an order ID."""
    q = query.lower()
    hits = [d for d in POLICY_DOCS if any(t in d["text"].lower() for t in q.split())]
    if not hits:
        hits = POLICY_DOCS[:1]
    return "\n".join(f"[{h['id']}] {h['text']}" for h in hits)


@tool
def search_products(query: str) -> str:
    """Search product catalog by name or SKU."""
    q = query.lower()
    hits = [p for p in PRODUCT_CATALOG if q in p["name"].lower() or q in p["sku"].lower()]
    return json.dumps(hits, default=str) if hits else "No products found."


@tool
def add_numbers(a: int, b: int) -> int:
    """Add two integers. Use only for explicit arithmetic requests."""
    return a + b


TOOLS = [lookup_order, search_policy, search_products, add_numbers]
TOOL_NODE = ToolNode(TOOLS)

SYSTEM_PROMPT = """You are an order support assistant for ShopCo.
Use lookup_order when the customer provides an order ID.
Use search_policy for refund and shipping policy questions.
Use search_products for catalog questions.
Cite policy doc IDs like [ret-02] when possible."""

print("Registered tools:", [t.name for t in TOOLS])

---

## 2. The ReAct Graph Pattern

| Piece | Role |
|-------|------|
| **`model` node** | Calls LLM with `bind_tools`; appends `AIMessage` to state |
| **`tools` node** | `ToolNode` runs every `tool_call`; appends `ToolMessage`s |
| **`tools_condition`** | If last message has `tool_calls` → `"tools"`, else → `END` |
| **`tools → model` edge** | After tools run, return to model for next decision |

State is **`MessagesState`** — a dict with a `messages` list that grows each step.

---

## 3. `ToolNode` in Isolation

You can test tool execution without any graph or LLM. Feed a synthetic `AIMessage` with `tool_calls` — `ToolNode` executes all calls (including **parallel** calls) and returns `ToolMessage`s.

In [ ]:
parallel_ai = AIMessage(
    content="",
    tool_calls=[
        {"name": "lookup_order", "args": {"order_id": "ORD-1001"}, "id": "call_order", "type": "tool_call"},
        {"name": "search_policy", "args": {"query": "returns refund"}, "id": "call_policy", "type": "tool_call"},
        {"name": "add_numbers", "args": {"a": 7, "b": 5}, "id": "call_add", "type": "tool_call"},
    ],
)

tool_result = TOOL_NODE.invoke({"messages": [parallel_ai]})

print(f"Input: 1 AIMessage with {len(parallel_ai.tool_calls)} tool_calls")
print(f"Output: {len(tool_result['messages'])} ToolMessages\n")
for msg in tool_result["messages"]:
    print(f"  [{msg.tool_call_id}] {str(msg.content)[:70]}...")

---

## 4. `tools_condition` Router

`tools_condition` inspects the **last message** in state:

- If it is an `AIMessage` **with** `tool_calls` → return `"tools"` (route to ToolNode).
- Otherwise → return `END`.

In [ ]:
print("Last message has tool_calls:", tools_condition({"messages": [parallel_ai]}))

final_ai = AIMessage(content="Order ORD-1001 is shipped. Returns allowed within 30 days [ret-02].")
print("Last message is final text:", tools_condition({"messages": [final_ai]}))

---

## 5. Offline Model Node (No API Key Required)

For learning and unit tests, replace the live LLM with a **rule-based model node** that returns realistic `AIMessage` objects. The graph structure stays identical — only the model node changes.

In [ ]:
def offline_model_node(state: MessagesState) -> dict[str, list[BaseMessage]]:
    """Simulate LLM: first turn → tool_calls; after tool results → final answer."""
    has_tool_results = any(isinstance(m, ToolMessage) for m in state["messages"])

    if not has_tool_results:
        user_text = next(m.content for m in state["messages"] if isinstance(m, HumanMessage))
        tool_calls = []
        if "ord" in user_text.lower() or "order" in user_text.lower():
            tool_calls.append({
                "name": "lookup_order", "args": {"order_id": "ORD-1001"},
                "id": "offline_order", "type": "tool_call",
            })
        if "return" in user_text.lower() or "refund" in user_text.lower() or "policy" in user_text.lower():
            tool_calls.append({
                "name": "search_policy", "args": {"query": "returns"},
                "id": "offline_policy", "type": "tool_call",
            })
        if "add" in user_text.lower() and any(c.isdigit() for c in user_text):
            tool_calls.append({
                "name": "add_numbers", "args": {"a": 7, "b": 5},
                "id": "offline_add", "type": "tool_call",
            })
        if not tool_calls:
            tool_calls.append({
                "name": "search_policy", "args": {"query": "shipping"},
                "id": "offline_default", "type": "tool_call",
            })
        return {"messages": [AIMessage(content="", tool_calls=tool_calls)]}

    snippets = [str(m.content)[:60] for m in state["messages"] if isinstance(m, ToolMessage)]
    answer = "Based on tool results: " + " | ".join(snippets)
    return {"messages": [AIMessage(content=answer)]}


def live_model_node(state: MessagesState) -> dict[str, list[BaseMessage]]:
    """Production model node — calls OpenAI with bind_tools."""
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    llm_with_tools = llm.bind_tools(TOOLS)
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

---

## 6. Compile the ReAct Graph

This is the standard LangGraph wiring for a tool-calling agent:

In [ ]:
def build_react_graph(use_live_llm: bool = False):
    """Build and compile the model ↔ tools ReAct loop."""
    model_fn = live_model_node if use_live_llm else offline_model_node

    builder = StateGraph(MessagesState)
    builder.add_node("model", model_fn)
    builder.add_node("tools", TOOL_NODE)

    builder.add_edge(START, "model")
    builder.add_conditional_edges("model", tools_condition)
    builder.add_edge("tools", "model")

    return builder.compile()


react_graph = build_react_graph(use_live_llm=USE_LIVE_LLM)

print("Graph nodes:", list(react_graph.get_graph().nodes.keys()))
print("Graph edges:", list(react_graph.get_graph().edges))

---

## 7. Invoke the Graph

Run the compiled graph with a user question. The offline model executes a full tool loop without an API key.

In [ ]:
def print_message_trace(messages: list[BaseMessage]) -> None:
    for i, m in enumerate(messages):
        name = type(m).__name__
        if isinstance(m, AIMessage) and m.tool_calls:
            tools_called = [tc["name"] for tc in m.tool_calls]
            print(f"  [{i}] {name}: tool_calls={tools_called}")
        else:
            preview = (m.content or "")[:75]
            print(f"  [{i}] {name}: {preview}")


user_question = "Where is order ORD-1001 and what is your return policy? Also add 7 and 5."

result = react_graph.invoke({"messages": [HumanMessage(content=user_question)]})

print(f"USER: {user_question}\n")
print("Message trace:")
print_message_trace(result["messages"])
print(f"\nFinal answer: {result['messages'][-1].content}")

---

## 8. Manual `while` Loop Equivalent

Before LangGraph, agents used a manual loop. The semantics are identical — LangGraph just makes routing, streaming, and checkpointing declarative.

In [ ]:
TOOLS_BY_NAME = {t.name: t for t in TOOLS}


def manual_react_loop(user_text: str, max_steps: int = 5) -> tuple[list[BaseMessage], list[str]]:
    """Manual while-loop — same logic as the LangGraph ReAct graph."""
    messages: list[BaseMessage] = [HumanMessage(content=user_text)]
    trace = []

    for step in range(max_steps):
        # model turn
        model_out = offline_model_node({"messages": messages})
        ai = model_out["messages"][0]
        messages.append(ai)
        trace.append(f"step {step}: model → {'tool_calls' if ai.tool_calls else 'final'}")

        if not ai.tool_calls:
            break

        # tools turn
        tool_out = TOOL_NODE.invoke({"messages": messages})
        messages.extend(tool_out["messages"])
        trace.append(f"step {step}: tools → {len(tool_out['messages'])} results")

    return messages, trace


manual_msgs, manual_trace = manual_react_loop("Where is order ORD-1001?")
print("Manual loop trace:")
for line in manual_trace:
    print(f"  {line}")
print(f"\nFinal: {manual_msgs[-1].content[:100]}...")

---

## 9. Graph vs Manual Loop

| Concern | LangGraph | Manual `while` |
|---------|-----------|----------------|
| **Checkpointing** | Built-in checkpointers | You build it |
| **Streaming** | `graph.stream()` per-node updates | Custom |
| **Routing** | `add_conditional_edges` | Nested `if` statements |
| **Visualization** | `get_graph().draw_mermaid()` | Log strings |
| **Human-in-the-loop** | `interrupt` API | Custom HTTP pause |
| **Simple demos** | Slightly heavier | Fine for learning |

**Rule of thumb:** use a manual loop to learn the semantics; move to LangGraph when you need persistence, branching, streaming, or multi-agent subgraphs.

---

## 10. Step Limits — Custom Router

`tools_condition` alone does not cap loops. Combine it with a **`step_count`** in state to prevent runaway cost.

In [ ]:
MAX_STEPS = 4


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    step_count: int


def counting_model_node(state: AgentState) -> dict:
    out = offline_model_node({"messages": state["messages"]})
    return {**out, "step_count": state.get("step_count", 0) + 1}


def should_continue(state: AgentState) -> Literal["tools", "__end__"]:
    if state.get("step_count", 0) >= MAX_STEPS:
        return END
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


limited_builder = StateGraph(AgentState)
limited_builder.add_node("model", counting_model_node)
limited_builder.add_node("tools", TOOL_NODE)
limited_builder.add_edge(START, "model")
limited_builder.add_conditional_edges("model", should_continue)
limited_builder.add_edge("tools", "model")
limited_graph = limited_builder.compile()

limited_result = limited_graph.invoke({
    "messages": [HumanMessage(content="Where is order ORD-1001 and what is the return policy?")],
    "step_count": 0,
})

print(f"Steps taken: {limited_result['step_count']} (max {MAX_STEPS})")
print(f"Messages: {len(limited_result['messages'])}")

---

## 11. Audited Tool Node — Logging Without Reimplementing

Wrap `ToolNode` instead of reimplementing it. Log tool names and arguments before delegation — useful for observability and compliance.

In [ ]:
AUDIT_LOG: list[str] = []


def audited_tool_node(state: MessagesState) -> dict:
    last = state["messages"][-1]
    if isinstance(last, AIMessage):
        for tc in last.tool_calls or []:
            AUDIT_LOG.append(f"invoke {tc['name']} args={tc['args']}")
    return TOOL_NODE.invoke(state)


AUDIT_LOG.clear()
_ = audited_tool_node({"messages": [parallel_ai]})

print("Audit log entries:")
for entry in AUDIT_LOG:
    print(f"  {entry}")

---

## 12. Tool Errors Stay in the Graph

`ToolNode` catches tool exceptions and returns the error text in a `ToolMessage`. The model can read the error and retry or explain the failure to the user.

In [ ]:
@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    return a / b


error_tool_node = ToolNode(TOOLS + [divide])

bad_ai = AIMessage(
    content="",
    tool_calls=[{"name": "divide", "args": {"a": 10, "b": 0}, "id": "err1", "type": "tool_call"}],
)

error_result = error_tool_node.invoke({"messages": [bad_ai]})
print("Tool error returned to model:")
print(error_result["messages"][0].content)

---

## 13. Streaming Graph Updates

`graph.stream()` emits per-node state updates — useful for showing progress in a UI. Set `USE_LIVE_LLM = True` for a live streaming demo.

In [ ]:
def stream_graph_updates(user_text: str) -> None:
    graph = build_react_graph(use_live_llm=USE_LIVE_LLM)
    print(f"Streaming updates for: {user_text!r}\n")
    for event in graph.stream(
        {"messages": [HumanMessage(content=user_text)]},
        stream_mode="updates",
    ):
        for node_name, update in event.items():
            msg_count = len(update.get("messages", []))
            print(f"  node={node_name!r} → {msg_count} new message(s)")


stream_graph_updates("Where is order ORD-1001?")

---

## 14. One Tool Node per Agent in Multi-Agent Systems

In multi-agent setups, give each specialist its **own small tool set** and **own ToolNode** — do not share one global ToolNode with 20 tools.

```
supervisor
    ├── order_agent graph  → ToolNode([lookup_order, search_policy])      # 2 tools
    ├── catalog_agent graph → ToolNode([search_products])                   # 1 tool
    └── chat_agent graph   → (no tools)
```

Each subgraph uses the same `model → tools_condition → tools → model` pattern but with a focused tool menu.

In [ ]:
AGENT_TOOLSETS = {
    "order_agent": [lookup_order, search_policy],
    "catalog_agent": [search_products],
    "math_agent": [add_numbers],
}


def build_agent_subgraph(agent_name: str, tools: list) -> Any:
    """Minimal per-agent ReAct subgraph with focused tools."""
    node = ToolNode(tools)
    builder = StateGraph(MessagesState)
    builder.add_node("model", offline_model_node)
    builder.add_node("tools", node)
    builder.add_edge(START, "model")
    builder.add_conditional_edges("model", tools_condition)
    builder.add_edge("tools", "model")
    return builder.compile()


subgraphs = {
    name: build_agent_subgraph(name, tools)
    for name, tools in AGENT_TOOLSETS.items()
}

for name, g in subgraphs.items():
    print(f"{name}: {len(AGENT_TOOLSETS[name])} tools, nodes={list(g.get_graph().nodes.keys())}")

---

## 15. Unit Testing `ToolNode` Without an LLM

Fast, deterministic tests: pass synthetic `AIMessage` objects directly to `ToolNode`. No graph compilation or API calls needed.

In [ ]:
TEST_CASES = [
    ("lookup_order", {"order_id": "ORD-1001"}, "shipped"),
    ("search_policy", {"query": "refund"}, "ret-02"),
    ("search_products", {"query": "laptop"}, "LAP-001"),
    ("add_numbers", {"a": 3, "b": 4}, "7"),
]

print(f"{'Tool':<18} {'Expected in output':<20} {'Pass'}")
print("-" * 50)

for tool_name, args, expected_substr in TEST_CASES:
    ai = AIMessage(
        content="",
        tool_calls=[{"name": tool_name, "args": args, "id": f"test_{tool_name}", "type": "tool_call"}],
    )
    out = TOOL_NODE.invoke({"messages": [ai]})
    content = str(out["messages"][0].content)
    passed = expected_substr in content
    print(f"{tool_name:<18} {expected_substr:<20} {'✓' if passed else '✗'}")

---

## 16. Optional — Live OpenAI Graph

Set `USE_LIVE_LLM = True` and re-run the build + invoke cells to use a real model.

In [ ]:
if USE_LIVE_LLM:
    live_graph = build_react_graph(use_live_llm=True)
    live_result = live_graph.invoke({
        "messages": [HumanMessage(content="What header carries JWT tokens? Search docs and add 7+5.")]
    })
    print_message_trace(live_result["messages"])
else:
    print("Offline mode active. The graph above runs with offline_model_node.")
    print("Set USE_LIVE_LLM = True in the setup cell for OpenAI.")

---

## 17. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| Forgot `tools → model` edge | Graph ends after one tool round | Add edge back to model |
| Wrong `tool_call_id` | 400 on next LLM call | Let `ToolNode` handle ids |
| Only executing first tool call | Missing parallel results | Use `ToolNode`, not manual `[0]` |
| No system prompt | Wrong tool selection | Add `SystemMessage` in model node |
| 20 tools on one agent | Frequent wrong-tool errors | Split into subgraphs with 2–5 tools each |
| No step limit | Runaway cost | Custom router with `step_count` |

---

## 18. Check Your Understanding

1. What does `ToolNode` accept as input and what does it return?
2. What are the two possible return values of `tools_condition`?
3. Why must there be an edge from `tools` back to `model`?
4. How is a LangGraph ReAct graph different from a manual `while` loop?
5. Why combine `tools_condition` with a custom `step_count` router?
6. Why should each agent in a multi-agent system have its own focused `ToolNode`?
7. How can you test tool execution without calling an LLM?

---

## 19. Summary

- **`ToolNode`** executes all `tool_calls` on the last `AIMessage` and appends `ToolMessage` results — including parallel calls.
- **`tools_condition`** routes to the tools node when `tool_calls` exist, otherwise to `END`.
- The standard ReAct graph: `START → model → tools_condition → tools → model → … → END`.
- Use an **offline model node** for learning and unit tests; swap in `bind_tools` + ChatOpenAI for production.
- Add **step limits** with custom state and a `should_continue` router.
- **Wrap** `ToolNode` for audit logging instead of reimplementing execution.
- In multi-agent systems, use **one focused ToolNode per specialist** (2–5 tools each).
- Prefer LangGraph over manual loops when you need checkpointing, streaming, branching, or human-in-the-loop gates.

You now have the standard LangGraph wiring for tool-calling agents — the same pattern used in production ReAct agents, with the graph making control flow explicit and testable.